# Generate Processing Configuration Suite

This notebook generates a set of configuration files for the EMI-Mesh pipeline to systematically test the impact of different image processing and meshing parameters on the final result.

## Objective
We want to evaluate how variations in resolution, morphological operation strength, and smoothing affect the resulting tetrahedral meshes of brain tissue.

## Configuration Design

The generated files follow these constraints:
- **Fixed Operations**: All files include `removeislands`, `dilate`, `smooth`, and `erode`.
- **Fixed removeislands**: `minsize=10000`.
- **Symmetry**: `dilate radius` == `erode radius` $\in [0, 4]$.
- **Smoothing**: `iterations` $\in [1, 4]$, `radius` $\in [1, 4]$.
- **Resolution**: `mip` $\in [0, 4]$, `dx` $\in [20, 200]$.
- **Envelop Size**: 20% to 50% of `dx`.

### Parameter Groups

#### Group A: Resolution & Scale (Constants: dil=2, ero=2, sm_it=1, sm_rad=2)
| mip | dx | env | Name Suffix |
| :--- | :--- | :--- | :--- |
| 0 | 20 | 4 | `mip0_dx20_env4` |
| 2 | 50 | 10 | `mip2_dx50_env10` |
| 4 | 100 | 20 | `mip4_dx100_env20` |
| 4 | 100 | 50 | `mip4_dx100_env50` |
| 4 | 200 | 40 | `mip4_dx200_env40` |
| 4 | 200 | 100 | `mip4_dx200_env100` |

#### Group B: Morphological Radius (Constants: mip=4, dx=100, env=50, sm_it=1, sm_rad=2)
| dil/ero radius | Name Suffix |
| :--- | :--- |
| 0 | `mip4_dx100_env50_rad0` |
| 1 | `mip4_dx100_env50_rad1` |
| 2 | `mip4_dx100_env50_rad2` |
| 3 | `mip4_dx100_env50_rad3` |
| 4 | `mip4_dx100_env50_rad4` |

#### Group C: Smoothing Strength (Constants: mip=4, dx=100, env=50, dil=2, ero=2)
| iterations | radius | Name Suffix |
| :--- | :--- | :--- |
| 1 | 1 | `mip4_dx100_env50_sm11` |
| 1 | 4 | `mip4_dx100_env50_sm14` |
| 4 | 1 | `mip4_dx100_env50_sm41` |
| 4 | 4 | `mip4_dx100_env50_sm44` |

In [ ]:
import yaml
import os
from pathlib import Path

# Output directory
output_dir = Path("../config_files/single_neuron_processing")
output_dir.mkdir(parents=True, exist_ok=True)

def create_config(mip, dx, env, dil_ero_rad, sm_it, sm_rad, suffix):
    # Construct the name based on project convention
    # format: mip{mip}_dx{dx}_rmis10000_dil{rad}_sm{it}{rad}_er{rad}_env{env}
    internal_name = f"mip{mip}_dx{dx}_rmis10000_dil{dil_ero_rad}_sm{sm_it}{sm_rad}_er{dil_ero_rad}_env{env}"
    
    config = {
        "name": internal_name,
        "raw": {
            "cloudpath": "precomputed://gs://iarpa_microns/minnie/minnie65/seg_m1300",
            "position": "0-0-0",
            "mip": mip,
            "size": 50000,
            "cell_type": "neuron",
            "cell_neuron_type": "4P",
            "cell_idx": 0,
            "cell_padding": 500,
            "cell_table_name": "aibs_metamodel_celltypes_v661",
            "cell_keep_surrounding": False
        },
        "processing": {
            "dx": dx,
            "operation": [
                f"removeislands minsize=10000",
                f"dilate radius={dil_ero_rad}",
                f"smooth iterations={sm_it} radius={sm_rad}",
                f"erode radius={dil_ero_rad}"
            ]
        },
        "meshing": {
            "envelopsize": env
        }
    }
    
    file_path = output_dir / f"neuron_{suffix}.yml"
    with open(file_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    return file_path

# Parameter sets
configs_to_generate = []

# Group A: Resolution & Scale (dil=2, ero=2, sm_it=1, sm_rad=2)
group_a = [
    (0, 20, 4, "mip0_dx20_env4"),
    (2, 50, 10, "mip2_dx50_env10"),
    (4, 100, 20, "mip4_dx100_env20"),
    (4, 100, 50, "mip4_dx100_env50"),
    (4, 200, 40, "mip4_dx200_env40"),
    (4, 200, 100, "mip4_dx200_env100"),
]
for mip, dx, env, suffix in group_a:
    configs_to_generate.append((mip, dx, env, 2, 1, 2, suffix))

# Group B: Morphological Radius (mip=4, dx=100, env=50, sm_it=1, sm_rad=2)
for rad in range(5):
    configs_to_generate.append((4, 100, 50, rad, 1, 2, f"mip4_dx100_env50_rad{rad}"))

# Group C: Smoothing Strength (mip=4, dx=100, env=50, dil=2, ero=2)
smooth_variations = [
    (1, 1, "sm11"),
    (1, 4, "sm14"),
    (4, 1, "sm41"),
    (4, 4, "sm44"),
]
for it, rad, suffix_part in smooth_variations:
    configs_to_generate.append((4, 100, 50, 2, it, rad, f"mip4_dx100_env50_{suffix_part}"))

# Generate files
generated_files = []
for params in configs_to_generate:
    path = create_config(*params)
    generated_files.append(path)

print(f"Successfully generated {len(generated_files)} configuration files in {output_dir}")
for f in generated_files:
    print(f)


## Summary
The configuration files have been generated and are ready for use in the EMI-Mesh pipeline. Each file corresponds to a specific parameter combination designed to test the sensitivity of the mesh quality to processing and meshing settings.